## ML Zoomcamp Homework 3
### Classification

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')

In [4]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [5]:
# 1. Check which columns have missing values
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
print("Columns with missing values:")
print(missing_values)

Columns with missing values:
lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64


In [7]:
print(f"Total missing values in the dataset: {df.isnull().sum().sum()}")

Total missing values in the dataset: 1194


In [8]:
for col in missing_values.index:
    missing_percentage = (missing_values[col] / len(df)) * 100
    print(f"{col}: {missing_percentage:.2f}% missing values")

lead_source: 2.96% missing values
industry: 4.80% missing values
employment_status: 3.88% missing values
location: 4.16% missing values
annual_income: 7.38% missing values
lead_score: 0.70% missing values


In [9]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [10]:
# Categorical columns 
cat_cols = ['lead_source', 'industry', 'employment_status', 'location']
df[cat_cols] = df[cat_cols].fillna('NA')

# Numerical columns 
num_cols = ['annual_income', 'lead_score']
df[num_cols] = df[num_cols].fillna(0.0)

In [11]:
print(df.isnull().sum())
print(df.dtypes)

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64
lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object


In [12]:
df['industry'].value_counts().head(1)

industry
technology    1173
Name: count, dtype: int64

## Q1

In [13]:
## Question 1
# What is the most frequent observation (mode) for the column industry?
print(df['industry'].mode()[0])

technology


In [14]:
df['industry'].mode()[0]

'technology'

In [15]:
df[['industry']]

,industry
0,technology
1,technology
2,retail
3,manufacturing
4,manufacturing
...,...
4995,education
4996,technology
4997,finance
4998,technology


## Q2

In [19]:
### Q2
df[['annual_income', 'number_of_courses_viewed',
    'interaction_count', 'lead_score']].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


interaction_count and lead_score: 0.91

In [21]:
### Split
df_full_train, df_test = train_test_split(df, test_size = 0.2, random_state = 42)
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 42)

In [22]:
y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']

del df_train['converted']
del df_val['converted']
del df_test['converted']

## Q3

Mutual information (MI) answers one question: how much does knowing feature X help me predict the target Y?

In [23]:
## Q3
from sklearn.metrics import mutual_info_score
categorical = ['industry', 'location', 'lead_source', 'employment_status']
for col in categorical:
    score = mutual_info_score(df_train[col], y_train)
    print(col, round(score, 2))

industry 0.0
location 0.0
lead_source 0.03
employment_status 0.02


lead_source with 0.03

## Q4

In [27]:
## Q4
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed',
             'interaction_count', 'lead_score']

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

# One-hot encode
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Train the model 
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predict on validation and computing accuracy
y_pred = model.predict(X_val)
accuracy = (y_pred == y_val).mean()

print(round(accuracy, 2))

0.64


In [28]:
df.shape

(5000, 9)

## Q5

In [29]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# All features
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed',
             'interaction_count', 'lead_score']
all_features = categorical + numerical

# A small helper so we don't repeat ourselves
def train_and_score(features):
    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    return (y_pred == y_val).mean()

# baseline accuracy with all features
baseline = train_and_score(all_features)
print('baseline:', baseline)
print()

# See how removing each feature affects the accuracy
for feature in all_features:
    reduced = [f for f in all_features if f != feature]
    acc = train_and_score(reduced)
    diff = baseline - acc
    print(f'{feature:28s} acc={acc:.3f}  diff={diff:.3f}')

baseline: 0.645

lead_source                  acc=0.642  diff=0.003
industry                     acc=0.645  diff=0.000
employment_status            acc=0.645  diff=0.000
location                     acc=0.645  diff=0.000
annual_income                acc=0.724  diff=-0.079
number_of_courses_viewed     acc=0.643  diff=0.002
interaction_count            acc=0.601  diff=0.044
lead_score                   acc=0.644  diff=0.001


number_of_courses_viewed, 0.002

## Q6

In [30]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed',
             'interaction_count', 'lead_score']
features = categorical + numerical

# Build the matrices once 
train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Try each C
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C,
                               max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    acc = (y_pred == y_val).mean()
    print(f'C={C:<10}  accuracy={round(acc, 3)}')

C=1e-06       accuracy=0.598
C=1e-05       accuracy=0.598
C=0.0001      accuracy=0.613
C=0.001       accuracy=0.645


C = 0.001